# H1.2 + Gate + cached visual features + VLM prompts
Attach the same three visual .h5 parts, MSCOCO/split, and BOTH complete VLM prompt .pt caches. Train one variant per Save & Run All version. Use objects first, then spatial. Both train from scratch with seed 42 and automatically evaluate all 5000 test images.


In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = Path('/kaggle/working/Image_Captioning')
def run(args):
    print('Running:', ' '.join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), check=True)
if not REPO.exists():
    run(['git', 'clone', 'https://github.com/Supzxjee/Image_Captioning.git', REPO])
os.chdir(REPO)
run(['git', 'fetch', 'origin'])
run(['git', 'checkout', 'main'])
run(['git', 'pull', '--ff-only'])
run(['git', 'rev-parse', 'HEAD'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])


In [ ]:
JSON = '/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json'
IMAGES = '/kaggle/input/datasets/vuthetam/mscoco-2014/images'
assert Path(JSON).is_file(), JSON
assert Path(IMAGES).is_dir(), IMAGES
VARIANT = 'objects'  # Then run a separate version with 'spatial'.
assert VARIANT in ('objects', 'spatial')
CACHE = Path('/kaggle/input/datasets/ducanh2403/visual-cache')
PROMPT_DIR = Path('/kaggle/input/your-vlm-prompt-cache')  # CHANGE to the attached prompt Dataset.
PROMPT = PROMPT_DIR / f'prompt_vlm_{VARIANT}.pt'
assert PROMPT.is_file(), PROMPT
assert len(list(CACHE.glob('visual_part_*.h5'))) == 3, CACHE
common = ['--dataset-json-path', JSON, '--base-path', IMAGES,
          '--prompt-cache-path', PROMPT, '--visual-cache', CACHE,
          '--visual-cache-id-key', 'coco_id',
          '--visual-preprocessing', 'bilinear', '--visual-precision', 'fp32']
run([sys.executable, '-u', 'train_h1_2_gated.py', '--mode', 'verify-cache',
     '--split', 'val', '--limit', 10, *common])
run([sys.executable, '-u', 'train_h1_2_gated.py', '--mode', 'train',
     '--epochs', 10, '--seed', 42, '--batch-size', 32, '--num-workers', 0,
     '--experiment-name', f'h1_2_gated_vlm_{VARIANT}', '--test-after-train', *common])
